# Grey-pixel White Balance Analysis

Uses the trained U-Net to predict grey pixels, estimates the scene illuminant from
those pixels, and applies white balance.  Results are compared against the
Gehler-Shi ground-truth illuminations using angular error.

Pipeline per image:
1. Load raw 12-bit linear PNG
2. Run trained model → grey-pixel probability map
3. Threshold → binary grey mask  (colorchecker region excluded)
4. Estimate illuminant = **median RGB of predicted grey pixels** in the raw image
5. Apply white balance: divide each channel by the estimated illuminant
6. Evaluate: angular error between estimated and ground-truth illuminant


In [ ]:
# %% 1. Imports & config
from __future__ import annotations

import inspect
import os
import random
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TQDM_DISABLE"] = "1"

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import cv2
import numpy as np
import scipy.io
import torch
import segmentation_models_pytorch as smp

# ── paths ─────────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path.cwd()
for p in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (p / "preprocess_canon1d.py").is_file():
        PROJECT_ROOT = p
        break

IMG_DIR       = Path(r"C:\Users\bauti\Desktop\IMLEX\UJM\Deep Learning\Project - Grey Color Detection\png_canon1d\cs\chroma\data\canon_dataset\568_dataset\png")
CC_DIR        = PROJECT_ROOT / "preprocessed_canon1d" / "cc_masks"
GT_ILLUM_FILE = PROJECT_ROOT / "groundtruth_568" / "real_illum_568..mat"
CHECKPOINT    = PROJECT_ROOT / "training" / "artifacts" / "best.pt"
OUT_DIR       = PROJECT_ROOT / "training" / "artifacts" / "wb_analysis"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── knobs ─────────────────────────────────────────────────────────────────────
PRED_THRESH   = 0.5    # grey-pixel probability threshold
MIN_GREY_PX   = 50     # skip illuminant estimate if fewer grey pixels found
DIVISIBLE_BY  = 32
SEED          = 42

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PROJECT_ROOT :", PROJECT_ROOT.resolve())
print("device       :", device)
print("checkpoint   :", CHECKPOINT)


## 2. Load model and ground-truth illuminations

In [ ]:
# ── model ────────────────────────────────────────────────────────────────────
_load_kw = {"map_location": device}
if "weights_only" in inspect.signature(torch.load).parameters:
    _load_kw["weights_only"] = False
ckpt = torch.load(CHECKPOINT, **_load_kw)

model = smp.Unet(encoder_name="resnet18", encoder_weights=None,
                 in_channels=3, classes=1, activation=None).to(device)
model.load_state_dict(ckpt["model"])
model.eval()
print(f"Loaded  epoch={ckpt.get('epoch')}  val_iou={ckpt.get('val_iou', 0):.4f}")

# ── ground-truth illuminations ────────────────────────────────────────────────
gt_illums = scipy.io.loadmat(str(GT_ILLUM_FILE))["real_rgb"]  # (568, 3)  [R, G, B]

# image list — same sorted order as used during preprocessing
all_images = sorted(IMG_DIR.glob("*.png"))
print(f"Images found : {len(all_images)}")
print(f"GT illums    : {gt_illums.shape}")


## 3. Helper functions

In [ ]:
def read_raw_rgb(path: Path) -> np.ndarray:
    # uint16 (H, W, 3) linear RGB
    bgr = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if bgr is None:
        raise FileNotFoundError(path)
    if bgr.ndim == 2:
        bgr = np.stack([bgr, bgr, bgr], axis=-1)
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB).astype(np.uint16)


def read_cc_mask(stem: str) -> np.ndarray:
    # float32 (H, W) -- 1 = colorchecker, 0 = scene
    path = CC_DIR / f"{stem}_cc_mask.png"
    if not path.is_file():
        return None
    g = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    return (g > 127).astype(np.float32)


def pad_to_divisible(img: np.ndarray, div: int = DIVISIBLE_BY):
    h, w = img.shape[:2]
    nh = (h + div - 1) // div * div
    nw = (w + div - 1) // div * div
    ph, pw = nh - h, nw - w
    if ph or pw:
        img = np.pad(img, ((0, ph), (0, pw), (0, 0)), mode="reflect")
    return img, (h, w)


@torch.no_grad()
def predict_grey_mask(img_raw_rgb: np.ndarray) -> np.ndarray:
    # Run model on raw uint16 image, return float32 (H, W) probability map
    img_f = (img_raw_rgb.astype(np.float32) / 4095.0).clip(0.0, 1.0)
    padded, (h0, w0) = pad_to_divisible(img_f)
    x = torch.from_numpy(padded).permute(2, 0, 1).unsqueeze(0).float().to(device)
    prob = torch.sigmoid(model(x))[0, 0, :h0, :w0].cpu().numpy()
    return prob


def estimate_illuminant(img_raw_rgb: np.ndarray, grey_mask: np.ndarray):
    # Estimate illuminant as median RGB of predicted grey pixels (or None if too few)
    px = img_raw_rgb[grey_mask > 0.5].astype(np.float64)  # (N, 3)
    if len(px) < MIN_GREY_PX:
        return None
    return np.median(px, axis=0)


def white_balance(img_raw_rgb: np.ndarray, illuminant: np.ndarray) -> np.ndarray:
    # Divide by illuminant and normalise to [0, 1]; returns float32 (H, W, 3)
    wb = img_raw_rgb.astype(np.float32) / (illuminant.astype(np.float32) + 1e-8)
    wb /= (wb.max() + 1e-8)
    return wb.clip(0.0, 1.0)


def angular_error(est: np.ndarray, gt: np.ndarray) -> float:
    # Angular error in degrees between two illuminant estimates
    e = est / (np.linalg.norm(est) + 1e-8)
    g = gt  / (np.linalg.norm(gt)  + 1e-8)
    return float(np.degrees(np.arccos(np.clip(np.dot(e, g), -1.0, 1.0))))


print("Helper functions defined.")


## 4. Run analysis over all images

In [ ]:
results = []   # one dict per image

print(f"{'#':>4}  {'Image':<22}  {'GreyPx':>7}  {'Grey%':>6}  {'EstIllum':>24}  {'GTIllum':>24}  {'AngErr':>7}")
print("-" * 100)

for idx, img_path in enumerate(all_images):
    stem   = img_path.stem
    raw    = read_raw_rgb(img_path)
    cc     = read_cc_mask(stem)

    # predict grey mask
    prob   = predict_grey_mask(raw)
    mask   = (prob > PRED_THRESH).astype(np.float32)

    # exclude colorchecker region
    if cc is not None:
        mask[cc > 0.5] = 0

    n_grey = int(mask.sum())
    grey_pct = n_grey / mask.size * 100

    # estimate illuminant from grey pixels
    est_illum = estimate_illuminant(raw, mask)

    # ground-truth illuminant
    gt_illum = gt_illums[idx]

    ang_err = angular_error(est_illum, gt_illum) if est_illum is not None else float("nan")

    results.append({
        "idx":        idx,
        "stem":       stem,
        "n_grey":     n_grey,
        "grey_pct":   grey_pct,
        "est_illum":  est_illum,
        "gt_illum":   gt_illum,
        "ang_err":    ang_err,
        "mask":       mask,        # kept for visualisation below
    })

    est_str = f"[{est_illum[0]:6.0f},{est_illum[1]:6.0f},{est_illum[2]:6.0f}]" if est_illum is not None else "    N/A (too few px)"
    gt_str  = f"[{gt_illum[0]:6.0f},{gt_illum[1]:6.0f},{gt_illum[2]:6.0f}]"
    print(f"{idx+1:>4}  {stem:<22}  {n_grey:>7}  {grey_pct:5.1f}%  {est_str}  {gt_str}  {ang_err:7.2f}")

print(f"\nProcessed {len(results)} images.")


## 5. Summary statistics

In [ ]:
valid = [r for r in results if not np.isnan(r["ang_err"])]
skipped = len(results) - len(valid)
errs = np.array([r["ang_err"] for r in valid])

print(f"{'='*50}")
print(f"  Angular error (degrees)  n={len(valid)}  skipped={skipped}")
print(f"{'='*50}")
print(f"  Mean   : {errs.mean():.3f}")
print(f"  Median : {np.median(errs):.3f}")
print(f"  Std    : {errs.std():.3f}")
print(f"  Min    : {errs.min():.3f}  ({valid[int(np.argmin(errs))]['stem']})")
print(f"  Max    : {errs.max():.3f}  ({valid[int(np.argmax(errs))]['stem']})")
print(f"  <5 deg : {(errs < 5).mean()*100:.1f}%")
print(f"  <10 deg: {(errs < 10).mean()*100:.1f}%")

# save angular error histogram
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(errs, bins=30, edgecolor="white", linewidth=0.5)
ax.axvline(errs.mean(),   color="red",    linestyle="--", label=f"mean {errs.mean():.2f}°")
ax.axvline(np.median(errs), color="orange", linestyle="--", label=f"median {np.median(errs):.2f}°")
ax.set_xlabel("Angular error (degrees)")
ax.set_ylabel("Images")
ax.set_title("Illuminant estimation error (grey-pixel method)")
ax.legend()
plt.tight_layout()
hist_path = OUT_DIR / "angular_error_histogram.png"
plt.savefig(hist_path, dpi=100)
plt.close()
print(f"\nHistogram saved to: {hist_path}")


## 6. Visual comparison: raw vs model-WB vs GT-WB

For a sample of images, saves side-by-side PNGs:
`raw | grey mask | model white-balanced | GT white-balanced`


In [ ]:
# pick a spread: best / median / worst angular error
valid_sorted = sorted(valid, key=lambda r: r["ang_err"])
n_total      = len(valid_sorted)
sample_idxs  = sorted({0, n_total // 4, n_total // 2, 3 * n_total // 4, n_total - 1})
sample       = [valid_sorted[i] for i in sample_idxs]

print(f"{'Image':<22}  {'AngErr':>7}  {'GreyPx':>7}  {'EstIllum':>24}  {'GTIllum':>24}")
print("-" * 95)

for r in sample:
    raw     = read_raw_rgb(IMG_DIR / f"{r['stem']}.png")
    mask    = r["mask"]
    est     = r["est_illum"]
    gt      = r["gt_illum"]

    # raw for display (gamma-corrected)
    raw_disp = np.power((raw.astype(np.float32) / 4095.0).clip(0, 1), 1 / 2.2)

    # model white balance
    wb_model = white_balance(raw, est) if est is not None else raw_disp
    wb_model_disp = np.power(wb_model.clip(0, 1), 1 / 2.2)

    # GT white balance
    wb_gt      = white_balance(raw, gt)
    wb_gt_disp = np.power(wb_gt.clip(0, 1), 1 / 2.2)

    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    axes[0].imshow(raw_disp);              axes[0].set_title("Raw (display γ)");             axes[0].axis("off")
    axes[1].imshow(mask, cmap="gray", vmin=0, vmax=1)
    axes[1].set_title(f"Grey mask  ({r['n_grey']} px  {r['grey_pct']:.1f}%)");              axes[1].axis("off")
    axes[2].imshow(wb_model_disp);         axes[2].set_title("Model WB (estimated illum)");  axes[2].axis("off")
    axes[3].imshow(wb_gt_disp);            axes[3].set_title("GT WB (ground truth illum)");   axes[3].axis("off")
    fig.suptitle(f"{r['stem']}  —  angular error = {r['ang_err']:.2f}°", fontsize=11)
    plt.tight_layout()

    out_path = OUT_DIR / f"{r['stem']}_wb_comparison.png"
    plt.savefig(out_path, dpi=90)
    plt.close()

    est_str = f"[{est[0]:6.0f},{est[1]:6.0f},{est[2]:6.0f}]" if est is not None else "N/A"
    gt_str  = f"[{gt[0]:6.0f},{gt[1]:6.0f},{gt[2]:6.0f}]"
    print(f"{r['stem']:<22}  {r['ang_err']:7.2f}  {r['n_grey']:>7}  {est_str}  {gt_str}")
    print(f"  saved -> {out_path}")
